# A. Project objective and dataset tables used

# B. Imports and configuration

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA_RAW = Path("/Users/derrick/projects/supply-chain-delivery-analytics/data_raw")
DATA_PROCESSED = Path("/Users/derrick/projects/supply-chain-delivery-analytics/data_processed")
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

# C. Load raw CSVs from data_raw

In [2]:
orders = pd.read_csv(DATA_RAW / "olist_orders_dataset.csv")
customers = pd.read_csv(DATA_RAW / "olist_customers_dataset.csv")
order_items = pd.read_csv(DATA_RAW / "olist_order_items_dataset.csv")
products = pd.read_csv(DATA_RAW / "olist_products_dataset.csv")
sellers = pd.read_csv(DATA_RAW / "olist_sellers_dataset.csv")
reviews = pd.read_csv(DATA_RAW / "olist_order_reviews_dataset.csv")

# D. Initial Quality Checks

### What we are looking for:
- datetime columns currently stored as objects
- missing delivery dates
- row counts
  
- whether the keys we need exist:
1. order_id
2. customer_id
3. seller_id
4. product_id

In [3]:
orders.shape, customers.shape, order_items.shape, products.shape, sellers.shape, reviews.shape

((99441, 8), (99441, 5), (112650, 7), (32951, 9), (3095, 4), (99224, 7))

In [4]:
orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [5]:
orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   order_id                       99441 non-null  str  
 1   customer_id                    99441 non-null  str  
 2   order_status                   99441 non-null  str  
 3   order_purchase_timestamp       99441 non-null  str  
 4   order_approved_at              99281 non-null  str  
 5   order_delivered_carrier_date   97658 non-null  str  
 6   order_delivered_customer_date  96476 non-null  str  
 7   order_estimated_delivery_date  99441 non-null  str  
dtypes: str(8)
memory usage: 6.1 MB


In [6]:
orders.isna().sum().sort_values(ascending = False)

order_delivered_customer_date    2965
order_delivered_carrier_date     1783
order_approved_at                 160
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_estimated_delivery_date       0
dtype: int64

In [7]:
order_items.info()
customers.info()
reviews.info()

<class 'pandas.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   order_id             112650 non-null  str    
 1   order_item_id        112650 non-null  int64  
 2   product_id           112650 non-null  str    
 3   seller_id            112650 non-null  str    
 4   shipping_limit_date  112650 non-null  str    
 5   price                112650 non-null  float64
 6   freight_value        112650 non-null  float64
dtypes: float64(2), int64(1), str(4)
memory usage: 6.0 MB
<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   customer_id               99441 non-null  str  
 1   customer_unique_id        99441 non-null  str  
 2   customer_zip_code_prefix  99441 non-null  int64
 3   customer_city    

In [8]:
reviews.isna().sum().sort_values(ascending = False)

review_comment_title       87656
review_comment_message     58247
review_id                      0
order_id                       0
review_score                   0
review_creation_date           0
review_answer_timestamp        0
dtype: int64

In [9]:
reviews_clean = reviews[["order_id", "review_score"]].drop_duplicates(subset=["order_id"])

In [10]:
print(reviews_clean.shape)
reviews_clean.head()

(98673, 2)


,order_id,review_score
0,73fc7af87114b39712e6da79b0a377eb,4
1,a548910a1c6147796b98fdf73dbeba33,5
2,f9e4b658b201a9f2ecdecbb34bed034b,5
3,658677c97b385a9be170737859d3511b,5
4,8e6bfb81e283fa7e4f11123a3fb894f1,5


# E. Datetime parsing

In [11]:
date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_cols:
    orders[col] = pd.to_datetime(orders[col], errors="coerce")

In [12]:
orders[date_cols].dtypes

order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

# F. Logistics feature engineering (delivery_days, delay_days, late_flag, etc.)

In [13]:
orders["delivered_flag"] = np.where(
    orders["order_delivered_customer_date"].notna(), 1, 0
)

orders["delivery_days"] = (
    orders["order_delivered_customer_date"] - orders["order_purchase_timestamp"]
).dt.days

orders["estimated_delivery_days"] = (
    orders["order_estimated_delivery_date"] - orders["order_purchase_timestamp"]
).dt.days

orders["delay_days"] = (
    orders["order_delivered_customer_date"] - orders["order_estimated_delivery_date"]
).dt.days

orders["delay_days"] = orders["delay_days"].clip(lower=0)

orders["late_flag"] = np.where(
    orders["order_delivered_customer_date"] > orders["order_estimated_delivery_date"],
    1,
    0
)

orders["purchase_year"] = orders["order_purchase_timestamp"].dt.year
orders["purchase_month"] = orders["order_purchase_timestamp"].dt.to_period("M").astype(str)
orders["purchase_dayofweek"] = orders["order_purchase_timestamp"].dt.day_name()

In [14]:
orders[[
    "order_id",
    "order_purchase_timestamp",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
    "delivery_days",
    "estimated_delivery_days",
    "delay_days",
    "late_flag"
]].head()

,order_id,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date,delivery_days,estimated_delivery_days,delay_days,late_flag
0,e481f51cbdc54678b7cc49136f2d6af7,2017-10-02 10:56:33,2017-10-10 21:25:13,2017-10-18,8.0,15,0.0,0
1,53cdb2fc8bc7dce0b6741e2150273451,2018-07-24 20:41:37,2018-08-07 15:27:45,2018-08-13,13.0,19,0.0,0
2,47770eb9100c2d0c44946d9cf07ec65d,2018-08-08 08:38:49,2018-08-17 18:06:29,2018-09-04,9.0,26,0.0,0
3,949d5b44dbf5de918fe9c16f97b45f8a,2017-11-18 19:28:06,2017-12-02 00:28:42,2017-12-15,13.0,26,0.0,0
4,ad21c59c0840e6cb83a9ceb5573f8159,2018-02-13 21:18:39,2018-02-16 18:17:02,2018-02-26,2.0,12,0.0,0


In [15]:
orders[["delivery_days", "estimated_delivery_days", "delay_days"]].describe()

,delivery_days,estimated_delivery_days,delay_days
count,96476.000000,99441.000000,96476.000000
mean,12.094086,23.403958,0.719391
std,9.551746,8.829562,4.652564
min,0.000000,1.000000,0.000000
25%,6.000000,18.000000,0.000000
50%,10.000000,23.000000,0.000000
75%,15.000000,28.000000,0.000000
max,209.000000,155.000000,188.000000


In [16]:
orders[orders["delivery_days"] < 0].shape

(0, 16)

In [17]:
orders[orders["estimated_delivery_days"] < 0].shape

(0, 16)

# G. Merge core tables into one analysis table

In [18]:
order_full = orders.merge(
    order_items,
    on="order_id",
    how="left"
)

order_full = order_full.merge(
    customers,
    on="customer_id",
    how="left"
)

order_full = order_full.merge(
    products,
    on="product_id",
    how="left"
)

order_full = order_full.merge(
    sellers,
    on="seller_id",
    how="left"
)

order_full = order_full.merge(
    reviews_clean,
    on="order_id",
    how="left"
)

In [19]:
order_full.shape

(113425, 38)

In [20]:
order_full.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivered_flag,delivery_days,...,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,seller_zip_code_prefix,seller_city,seller_state,review_score
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,1,8.0,...,268.0,4.0,500.0,19.0,8.0,13.0,9350.0,maua,SP,4.0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,1,13.0,...,178.0,1.0,400.0,19.0,13.0,19.0,31570.0,belo horizonte,SP,4.0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,1,9.0,...,232.0,1.0,420.0,24.0,19.0,21.0,14840.0,guariba,SP,5.0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,1,13.0,...,468.0,3.0,450.0,30.0,10.0,20.0,31842.0,belo horizonte,MG,5.0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,1,2.0,...,316.0,4.0,250.0,51.0,15.0,15.0,8752.0,mogi das cruzes,SP,5.0


In [21]:
order_full.isna().sum().sort_values(ascending = False)

delivery_days                    3229
delay_days                       3229
order_delivered_customer_date    3229
product_category_name            2378
product_name_lenght              2378
product_description_lenght       2378
product_photos_qty               2378
order_delivered_carrier_date     1968
review_score                      961
product_length_cm                 793
product_height_cm                 793
product_weight_g                  793
product_width_cm                  793
price                             775
seller_zip_code_prefix            775
seller_city                       775
seller_state                      775
freight_value                     775
shipping_limit_date               775
product_id                        775
order_item_id                     775
seller_id                         775
order_approved_at                 161
customer_id                         0
late_flag                           0
order_status                        0
order_purcha

# H. Order level cost/value

In [22]:
order_full["freight_ratio"] = order_full["freight_value"] / order_full["price"]
order_full["freight_ratio"] = order_full["freight_ratio"].replace([np.inf, -np.inf], np.nan)

In [23]:
order_full["price_band"] = pd.qcut(
    order_full["price"],
    q=4,
    labels=["Low", "Medium", "High", "Premium"]
)

In [24]:
order_full[["price", "freight_value", "freight_ratio", "price_band"]].head()

,price,freight_value,freight_ratio,price_band
0,29.99,8.72,0.290764,Low
1,118.70,22.76,0.191744,High
2,159.90,19.22,0.120200,Premium
3,45.00,27.20,0.604444,Medium
4,19.90,8.72,0.438191,Low


In [25]:
order_full.isna().sum().sort_values(ascending = False).head(30)

delivery_days                    3229
order_delivered_customer_date    3229
delay_days                       3229
product_name_lenght              2378
product_description_lenght       2378
product_photos_qty               2378
product_category_name            2378
order_delivered_carrier_date     1968
review_score                      961
product_width_cm                  793
product_height_cm                 793
product_length_cm                 793
product_weight_g                  793
seller_city                       775
seller_zip_code_prefix            775
freight_value                     775
seller_state                      775
freight_ratio                     775
price                             775
price_band                        775
seller_id                         775
product_id                        775
order_item_id                     775
shipping_limit_date               775
order_approved_at                 161
customer_id                         0
late_flag   

In [26]:
# percentage of missing values
(order_full.isna().mean().sort_values(ascending=False) * 100).round(2).head(30)

delivery_days                    2.85
order_delivered_customer_date    2.85
delay_days                       2.85
product_name_lenght              2.10
product_description_lenght       2.10
product_photos_qty               2.10
product_category_name            2.10
order_delivered_carrier_date     1.74
review_score                     0.85
product_width_cm                 0.70
product_height_cm                0.70
product_length_cm                0.70
product_weight_g                 0.70
seller_city                      0.68
seller_zip_code_prefix           0.68
freight_value                    0.68
seller_state                     0.68
freight_ratio                    0.68
price                            0.68
price_band                       0.68
seller_id                        0.68
product_id                       0.68
order_item_id                    0.68
shipping_limit_date              0.68
order_approved_at                0.14
customer_id                      0.00
late_flag   

In [27]:
# checking key fields 

order_full[[
    "order_id",
    "customer_id",
    "seller_id",
    "product_id",
    "delivery_days",
    "estimated_delivery_days",
    "delay_days",
    "late_flag",
    "review_score",
    "price",
    "freight_value"
]].isna().sum()

order_id                      0
customer_id                   0
seller_id                   775
product_id                  775
delivery_days              3229
estimated_delivery_days       0
delay_days                 3229
late_flag                     0
review_score                961
price                       775
freight_value               775
dtype: int64

In [28]:
# Drop rows missing core identifiers
order_full = order_full.dropna(subset=["order_id", "customer_id", "product_id", "seller_id"])

# Handle product category metadata
order_full["product_category_name"] = order_full["product_category_name"].fillna("unknown_category")

# Handle freight and price issues
order_full = order_full[order_full["price"].notna()]
order_full["freight_value"] = order_full["freight_value"].fillna(order_full["freight_value"].median())

# Reviews: keep missing (no imputation)

In [29]:
order_full.isna().sum().sort_values(ascending=False).head(20)

delay_days                       2454
order_delivered_customer_date    2454
delivery_days                    2454
product_name_lenght              1603
product_description_lenght       1603
product_photos_qty               1603
order_delivered_carrier_date     1194
review_score                      942
product_weight_g                   18
product_width_cm                   18
product_height_cm                  18
product_length_cm                  18
order_approved_at                  15
customer_state                      0
product_category_name               0
order_id                            0
customer_zip_code_prefix            0
seller_zip_code_prefix              0
seller_city                         0
seller_state                        0
dtype: int64

In [30]:
# checking key fields 

order_full[[
    "order_id",
    "customer_id",
    "seller_id",
    "product_id",
    "delivery_days",
    "estimated_delivery_days",
    "delay_days",
    "late_flag",
    "review_score",
    "price",
    "freight_value"
]].isna().sum()

order_id                      0
customer_id                   0
seller_id                     0
product_id                    0
delivery_days              2454
estimated_delivery_days       0
delay_days                 2454
late_flag                     0
review_score                942
price                         0
freight_value                 0
dtype: int64

In [31]:
# Check order status distribution for rows with missing delivery metrics
order_full.loc[
    order_full["delivery_days"].isna(),
    "order_status"
].value_counts(dropna=False)

order_status
shipped        1185
canceled        535
invoiced        359
processing      357
delivered         8
unavailable       7
approved          3
Name: count, dtype: int64

In [32]:
# Delivered flag already exists from orders table, but ensure it is clean
order_full["delivered_flag"] = np.where(
    order_full["order_delivered_customer_date"].notna(), 1, 0
)

# Review submitted flag
order_full["review_submitted_flag"] = np.where(
    order_full["review_score"].notna(), 1, 0
)

# Missing delivery metric flag
order_full["missing_delivery_metrics_flag"] = np.where(
    order_full["delivery_days"].isna(), 1, 0
)

In [33]:
order_full["late_flag"] = np.where(
    order_full["order_delivered_customer_date"].notna(),
    np.where(
        order_full["order_delivered_customer_date"] > order_full["order_estimated_delivery_date"],
        1,
        0
    ),
    np.nan
)

In [34]:
order_full[[
    "delivery_days",
    "delay_days",
    "review_score",
    "delivered_flag",
    "review_submitted_flag",
    "late_flag"
]].isna().sum()

delivery_days            2454
delay_days               2454
review_score              942
delivered_flag              0
review_submitted_flag       0
late_flag                2454
dtype: int64

In [35]:
delivered_orders = order_full[order_full["delivered_flag"] == 1].copy()

In [36]:
delivered_orders[[
    "delivery_days",
    "delay_days",
    "late_flag"
]].isna().sum()

delivery_days    0
delay_days       0
late_flag        0
dtype: int64

In [37]:
reviewed_orders = order_full[order_full["review_submitted_flag"] == 1].copy()

# I. Export cleaned dataset to data_processed/orders_cleaned.csv

In [38]:
order_full.to_csv(DATA_PROCESSED / "orders_cleaned.csv", index=False)
delivered_orders.to_csv(DATA_PROCESSED / "delivered_orders_cleaned.csv", index=False)
reviewed_orders.to_csv(DATA_PROCESSED / "reviewed_orders_cleaned.csv", index=False)

In [39]:
sorted([p.name for p in DATA_PROCESSED.glob("*.csv")])

['delivered_orders_cleaned.csv',
 'orders_cleaned.csv',
 'reviewed_orders_cleaned.csv']

In [40]:
order_full.loc[
    order_full["delivery_days"].isna(),
    "order_status"
].value_counts()

order_status
shipped        1185
canceled        535
invoiced        359
processing      357
delivered         8
unavailable       7
approved          3
Name: count, dtype: int64

In [41]:
order_full.loc[
    (order_full["order_status"] == "delivered") & (order_full["delivery_days"].isna()),
    [
        "order_id",
        "order_status",
        "order_purchase_timestamp",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
        "delivery_days",
        "delay_days"
    ]
]

,order_id,order_status,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date,delivery_days,delay_days
3376,2d1e2d5bf4dc7227b3bfebb81328c15f,delivered,2017-11-28 17:44:07,NaT,2017-12-18,NaN,NaN
23485,f5dd62b788049ad9fc0526e3ad11a097,delivered,2018-06-20 06:58:43,NaT,2018-07-16,NaN,NaN
49966,2ebdfc4f15f23b91474edf87475f108e,delivered,2018-07-01 17:05:11,NaT,2018-07-30,NaN,NaN
90294,e69f75a717d64fc5ecdfae42b2e8e086,delivered,2018-07-01 22:05:55,NaT,2018-07-30,NaN,NaN
94388,0d3268bad9b086af767785e3f0fc0133,delivered,2018-07-01 21:14:02,NaT,2018-07-24,NaN,NaN
105606,2d858f451373b04fb5c984a1cc2defaf,delivered,2017-05-25 23:22:43,NaT,2017-06-23,NaN,NaN
111339,ab7c89dc1bf4a1ead9d6ec1ec8968a84,delivered,2018-06-08 12:09:39,NaT,2018-06-26,NaN,NaN
111783,20edc82cf5400ce95e1afacc25798b31,delivered,2018-06-27 16:09:12,NaT,2018-07-19,NaN,NaN


### Breakdown:
1. shipped, invoiced, processing, approved = still in the pipeline
2. canceled, unavailable = never fulfilled
3. delivered = only 8 rows, likely data inconsistency or edge cases

So the correct move is:
- keep those rows in the full dataset
- exclude non-delivered rows from delivery KPI calculations
- inspect the 8 delivered rows with missing delivery metrics

In [42]:
# Rebuild delivered flag using both status and actual delivery timestamp
order_full["delivered_flag"] = np.where(
    (order_full["order_status"] == "delivered") &
    (order_full["order_delivered_customer_date"].notna()),
    1,
    0
)

# Review submitted flag
order_full["review_submitted_flag"] = np.where(
    order_full["review_score"].notna(),
    1,
    0
)

# Late flag should only exist for valid delivered orders
order_full["late_flag"] = np.where(
    order_full["delivered_flag"] == 1,
    np.where(
        order_full["order_delivered_customer_date"] > order_full["order_estimated_delivery_date"],
        1,
        0
    ),
    np.nan
)

In [43]:
# check for missing values again
delivered_orders[[
    "delivery_days",
    "delay_days",
    "late_flag"
]].isna().sum()

delivery_days    0
delay_days       0
late_flag        0
dtype: int64

In [44]:
delivered_orders.shape

(110196, 42)

In [45]:
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

order_full.to_csv(DATA_PROCESSED / "orders_cleaned.csv", index=False)
delivered_orders.to_csv(DATA_PROCESSED / "delivered_orders_cleaned.csv", index=False)

In [46]:
sorted([p.name for p in DATA_PROCESSED.glob("*.csv")])

['delivered_orders_cleaned.csv',
 'orders_cleaned.csv',
 'reviewed_orders_cleaned.csv']

In [47]:
print(order_full.shape)
print(delivered_orders.shape)
print(delivered_orders["late_flag"].isna().sum())

(112650, 42)
(110196, 42)
0
